## Step 1 — Check uploaded files

In this step, I check which files are available inside the Colab `/content` folder.
This helps confirm that both the original video file and the Persian voice/audio file were uploaded correctly before processing them.

The output shows the file names that can be used later in the code.


In [ ]:
import os

print(os.listdir("/content"))

['.config', '202606062328.mp4', 'WhatsApp Audio 2026-06-06 at 11.48.40 PM.mp4', 'sample_data']


## Step 2 — Define a function to get media duration

In this step, I use `ffprobe` to calculate the duration of a video or audio file in seconds.

This function is important because the original video and the Persian voice file may not have the same length.
By measuring both durations, I can decide whether to speed up the audio or slow down the video so that they match correctly.


In [ ]:
import subprocess

def get_duration(file):
    result = subprocess.run(
        [
            "ffprobe", "-v", "error",
            "-show_entries", "format=duration",
            "-of", "default=noprint_wrappers=1:nokey=1",
            file
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )
    return float(result.stdout.strip())

video_duration = get_duration("/content/202606062328.mp4")
voice_duration = get_duration("/content/WhatsApp Audio 2026-06-06 at 11.48.40 PM.mp4")

print("Video duration:", video_duration, "seconds")
print("Voice duration:", voice_duration, "seconds")

Video duration: 27.818 seconds
Voice duration: 41.450667 seconds


## Step 3 — Set file paths and calculate durations

In this step, I define the path of the original video and the Persian audio file.

Then I calculate the duration of each file.
The result shows that the original video is shorter than the Persian voice recording, so the two files need to be synchronized before creating the final video.



## Step 4 — Calculate the synchronization ratio

In this step, I calculate the ratio between the Persian voice duration and the original video duration.

The Persian voice is about 1.49 times longer than the video.  
This means that if I want to keep the Persian voice natural, I need to slow down the video instead of speeding up the voice.

In [ ]:
import subprocess

video_path = "/content/202606062328.mp4"
voice_path = "/content/WhatsApp Audio 2026-06-06 at 11.48.40 PM.mp4"

def get_duration(file):
    result = subprocess.run(
        [
            "ffprobe", "-v", "error",
            "-show_entries", "format=duration",
            "-of", "default=noprint_wrappers=1:nokey=1",
            file
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )
    return float(result.stdout.strip())

video_duration = get_duration(video_path)
voice_duration = get_duration(voice_path)

speed = voice_duration / video_duration

print("Video duration:", video_duration)
print("Voice duration:", voice_duration)
print("Required audio speed:", speed)

Video duration: 27.818
Voice duration: 41.450667
Required audio speed: 1.4900663958587965


In [ ]:
video_path = "/content/202606062328.mp4"
voice_path = "/content/WhatsApp Audio 2026-06-06 at 11.48.40 PM.mp4"
output_path = "/content/final_persian_video.mp4"

speed = 41.450667 / 27.818

!ffmpeg -y \
-i "$video_path" \
-i "$voice_path" \
-map 0:v:0 \
-map 1:a:0 \
-c:v copy \
-c:a aac \
-filter:a "atempo=$speed,volume=1.5" \
-shortest \
"$output_path"

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [ ]:
from IPython.display import Video

Video("/content/final_persian_video.mp4", embed=True)

## Step 5 — Replace the original audio with Persian voice

In this step, I use FFmpeg to create the final video.

The original English audio is removed, and the Persian voice file is added instead.  
Because the Persian voice is longer than the video, the video speed is slowed down using `setpts`.

This method keeps the Persian voice more natural and avoids making the speech sound too fast.

In [ ]:
video_path = "/content/202606062328.mp4"
voice_path = "/content/WhatsApp Audio 2026-06-06 at 11.48.40 PM.mp4"
output_path = "/content/final_natural_voice_video.mp4"

video_duration = 27.818
voice_duration = 41.450667

slow_factor = voice_duration / video_duration

print("Video slow factor:", slow_factor)

!ffmpeg -y \
-i "$video_path" \
-i "$voice_path" \
-filter:v "setpts=$slow_factor*PTS" \
-map 0:v:0 \
-map 1:a:0 \
-c:v libx264 \
-c:a aac \
-shortest \
"$output_path"

Video slow factor: 1.4900663958587965
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --e

In [ ]:
from IPython.display import Video

Video("/content/final_natural_voice_video.mp4", embed=True)

# Persian Voice Replacement for an English Video

This notebook replaces the original English audio of a video with a Persian voice recording.

The workflow includes:
1. Checking uploaded files
2. Measuring video and audio duration
3. Calculating the synchronization ratio
4. Slowing down the video to match the Persian voice
5. Replacing the original audio with Persian audio
6. Previewing and downloading the final video

# ***Next***

## Note about TTS installation in Colab

I first tried to install the `TTS` library in Google Colab using:

`pip install TTS pydub`

However, the installation failed because the current Colab Python version is not compatible with the available `TTS` package versions.

The error message shows that some versions of `TTS` require an older Python version, such as Python below 3.9. Because of this compatibility issue, the `TTS` package could not be installed directly in this Colab environment.

As a result, I did not continue with this method for generating Persian speech inside Colab. Instead, I used an external Persian voice/audio file and processed it with FFmpeg to replace the original English audio in the video.

In [ ]:
!pip install -q TTS pydub

ERROR: Ignored the following versions that require a different python version: 0.0.10.2 Requires-Python >=3.6.0, <3.9; 0.0.10.3 Requires-Python >=3.6.0, <3.9; 0.0.11 Requires-Python >=3.6.0, <3.9; 0.0.12 Requires-Python >=3.6.0, <3.9; 0.0.13.1 Requires-Python >=3.6.0, <3.9; 0.0.13.2 Requires-Python >=3.6.0, <3.9; 0.0.14.1 Requires-Python >=3.6.0, <3.9; 0.0.15 Requires-Python >=3.6.0, <3.9; 0.0.15.1 Requires-Python >=3.6.0, <3.9; 0.0.9 Requires-Python >=3.6.0, <3.9; 0.0.9.1 Requires-Python >=3.6.0, <3.9; 0.0.9.2 Requires-Python >=3.6.0, <3.9; 0.0.9a10 Requires-Python >=3.6.0, <3.9; 0.0.9a9 Requires-Python >=3.6.0, <3.9; 0.1.0 Requires-Python >=3.6.0, <3.10; 0.1.1 Requires-Python >=3.6.0, <3.10; 0.1.2 Requires-Python >=3.6.0, <3.10; 0.1.3 Requires-Python >=3.6.0, <3.10; 0.10.0 Requires-Python >=3.7.0, <3.11; 0.10.1 Requires-Python >=3.7.0, <3.11; 0.10.2 Requires-Python >=3.7.0, <3.11; 0.11.0 Requires-Python >=3.7.0, <3.11; 0.11.1 Requires-Python >=3.7.0, <3.11; 0.12.0 Requires-Python >=3

# ***Main*** work

## Text-to-Speech Voice Cloning Test

In this step, I tested the Coqui XTTS multilingual model to generate speech from Persian text using my own voice as a reference.

The workflow was:

1. Install the required TTS library.
2. Check that CUDA/GPU is available.
3. Convert my reference voice file to WAV format.
4. Load the XTTS multilingual model.
5. Provide Persian text as input.
6. Use my voice sample as the speaker reference.
7. Generate a new audio file.

Although the model successfully generated an audio file, the output did not sound like natural Persian. The pronunciation sounded closer to Arabic because Persian/Farsi is not properly supported by this model in the same way as supported languages.

In the code, I used:

language="ar"

because Persian/Farsi was not available as a supported language option. This allowed the model to process the text, but it caused the pronunciation to follow Arabic phonetic patterns instead of Persian.

Therefore, this method was not suitable for generating high-quality Persian narration or Persian voice cloning.

In [ ]:
!python --version

Python 3.12.13


In [ ]:
!pip install -q coqui-tts pydub

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 862.8/862.8 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 997.3/997.3 kB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 648.4/648.4 kB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 7.2 MB/s eta 0:00:00


restart

In [ ]:
!pip uninstall -y transformers
!pip install -q transformers==4.57.6

Found existing installation: transformers 5.9.0
Uninstalling transformers-5.9.0:
  Successfully uninstalled transformers-5.9.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 91.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 45.0 MB/s eta 0:00:00


restart

In [ ]:
import torch
from TTS.api import TTS

print("CUDA:", torch.cuda.is_available())
print("TTS import OK")

CUDA: True
TTS import OK


Important limitation

The XTTS model used in this experiment does not properly support Persian/Farsi voice generation.

Since Persian/Farsi was not available as a language option, I tested the closest possible option by using Arabic:

language="ar"

However, this caused the generated speech to sound Arabic rather than Persian.  
As a result, the model could not generate a natural Persian voice, even though the input text was Persian and the reference voice was my own voice.

In [ ]:
import os
import torch
from TTS.api import TTS

os.environ["COQUI_TOS_AGREED"] = "1"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# 1) Convert your voice reference to wav
voice_mp4 = "/content/WhatsApp Audio 2026-06-06 at 11.48.40 PM.mp4"
voice_wav = "/content/my_voice_reference.wav"

!ffmpeg -y -i "$voice_mp4" -vn -ac 1 -ar 22050 "$voice_wav"

# 2) Persian text
persian_text = """
وقتی برای نگاه کردن به گوشی، سرتان را به جلو خم می‌کنید،
حدود هجده کیلوگرم فشار اضافی به گردن وارد می‌شود.

این فشار اضافی را عضلات گردن تحمل می‌کنند
تا سر را در همان حالت نگه دارند.

اگر دقیق‌تر به عضلات نگاه کنیم،
می‌بینیم این فشار به عضله ذوزنقه‌ای منتقل می‌شود،
و همزمان به استخوان‌ها و مفاصل گردن هم فشار وارد می‌کند.

پس دفعه بعد که مشغول اسکرول کردن هستید،
این موضوع را به خاطر داشته باشید.
"""

# 3) Load model
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)

# 4) Generate voice
output_audio = "/content/ai_persian_voice.wav"

tts.tts_to_file(
    text=persian_text,
    speaker_wav=voice_wav,
    language="ar",   # مهم: fa پشتیبانی نمی‌شود، فعلاً ar
    file_path=output_audio,
    split_sentences=True
)

print("Done:", output_audio)

Device: cuda
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-

## Audio output review

The generated audio file was successfully created and could be played inside Colab.

However, after listening to the result, I found that the pronunciation was not correct for Persian. The voice sounded more like Arabic because the model used Arabic language settings instead of native Persian support.

For this reason, I decided not to use this generated audio as the final narration. Instead, I used a separate Persian voice recording and synchronized it with the video using FFmpeg.

## Conclusion

This experiment showed that the Coqui XTTS model can perform multilingual voice cloning, but it is not reliable for Persian/Farsi narration in this setup.

The main issue was language support. Since Persian was not available as a supported language, using Arabic produced incorrect pronunciation.

Final decision:
I did not use the AI-generated audio. I used my own Persian audio recording and replaced the original English audio in the video with FFmpeg.

In [ ]:
from IPython.display import Audio

Audio("/content/ai_persian_voice.wav")

# ***Next***

## Step 1 — Check Python version

Before installing the text-to-speech package, I checked the Python version used by Google Colab.

This is important because some machine learning and text-to-speech libraries only work with specific Python versions. In this notebook, Colab is using Python 3.12.13.

Previously, the `TTS` package had compatibility issues with this Python version, so I tested another package called `edge-tts`, which supports Persian voices.

In [ ]:
!python --version

Python 3.12.13


Step 2 — Install Edge TTS
In this step, I installed the edge-tts package.

edge-tts is a text-to-speech tool that can generate speech using Microsoft Edge online voices. Unlike the previous Coqui TTS method, this tool includes Persian/Iranian voices, so it is more suitable for generating Persian narration.

I also used the quiet installation option -q to reduce unnecessary installation output in the notebook.

In [ ]:
!pip install -q edge-tts

## Step 3 — Check available Persian voices

After installing `edge-tts`, I checked whether Persian/Iranian voices were available.

The command lists all voices and filters the result using `fa-IR`, which represents Persian/Iran. The output shows two supported Persian voices:

- `fa-IR-DilaraNeural` — female Persian voice
- `fa-IR-FaridNeural` — male Persian voice

This confirms that Persian text-to-speech is supported in this method.

In [ ]:
!edge-tts --list-voices | grep fa-IR

fa-IR-DilaraNeural                 Female    General                Friendly, Positive
fa-IR-FaridNeural                  Male      General                Friendly, Positive


## Step 4 — Prepare the Persian narration text

In this step, I wrote the Persian narration text that will be converted into speech.

The text explains how bending the head forward while using a phone can increase pressure on the neck. I saved this text into a `.txt` file using UTF-8 encoding.

Using UTF-8 is important because Persian characters need proper encoding to be read correctly by the text-to-speech tool.

In [ ]:
persian_text = """
وقتی برای نگاه کردن به گوشی، سرتان را جلو می‌آورید،
فشار زیادی به گردن وارد می‌شود.

این فشار را عضلات گردن تحمل می‌کنند
تا سر را در همان حالت نگه دارند.

بخشی از این فشار به عضله ذوزنقه‌ای منتقل می‌شود،
و همزمان استخوان‌ها و مفاصل گردن را هم درگیر می‌کند.

پس دفعه بعد که اسکرول می‌کنید،
حالت گردنتان را به خاطر داشته باشید.
"""

with open("/content/persian_text.txt", "w", encoding="utf-8") as f:
    f.write(persian_text)

## Step 5 — Generate Persian AI voic ,Test different speech rates

In this step, I tested two different speech rates for the Persian narration using `edge-tts`.

First, I tested `--rate=-5%`, which makes the voice slightly slower than the default speed.

Then, I tested `--rate=-10%`, which makes the voice slower and more suitable for a clear educational narration.

After listening to both outputs, I selected the `-10%` version because it sounded more natural and easier to understand.

In [ ]:
!edge-tts \
--voice fa-IR-DilaraNeural \
--rate=-5% \
--text "$(cat /content/persian_text.txt)" \
--write-media /content/persian_ai_voice.mp3

In [ ]:
!edge-tts \
--voice fa-IR-DilaraNeural \
--rate=-10% \
--text "$(cat /content/persian_text.txt)" \
--write-media /content/persian_ai_voice.mp3

## Step 6 — Preview the generated Persian voice

In this step, I played the generated Persian audio inside Colab.

This allows me to check the pronunciation, speed, and overall quality before combining it with the video.

The first generated audio was around 30 seconds long, which was slightly longer than the original video.

In [ ]:
from IPython.display import Audio

Audio("/content/persian_ai_voice.mp3")

## Step 7 — Define a function to measure media duration

In this step, I created a helper function called `get_duration`.

This function uses `ffprobe` to read the duration of a video or audio file in seconds. It is useful because the original video and the generated Persian audio must have similar durations before they are combined.

If the audio is longer than the video, I need to speed up the audio slightly. If the audio is shorter, I may need to slow it down or adjust the text-to-speech rate.

In [ ]:
import subprocess

def get_duration(file):
    result = subprocess.run(
        [
            "ffprobe", "-v", "error",
            "-show_entries", "format=duration",
            "-of", "default=noprint_wrappers=1:nokey=1",
            file
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )
    return float(result.stdout.strip())

video_path = "/content/202606062328.mp4"
audio_path = "/content/persian_ai_voice.mp3"

print("Video duration:", get_duration(video_path))
print("Audio duration:", get_duration(audio_path))

Video duration: 27.818
Audio duration: 30.984


## Step 8 — Compare the original video duration and generated audio duration

In this step, I measured the duration of the original video and the generated Persian audio.

The original video duration was about 27.818 seconds.  
The generated Persian audio duration was about 30.984 seconds.

This means the audio was longer than the video, so it needed to be slightly accelerated to fit the video duration.

## Step 9 — Calculate the required audio speed

In this step, I calculated how much the Persian audio needs to be sped up to match the video length.

The formula is:

audio duration ÷ video duration

The result was about 1.11. This means the audio needs to play about 11% faster to fit the original video duration.

This adjustment is small, so the voice should still sound natural.

In [ ]:
video_path = "/content/202606062328.mp4"
audio_path = "/content/persian_ai_voice.mp3"

video_duration = 27.818
audio_duration = 30.984

speed = audio_duration / video_duration
print("Speed needed:", speed)

Speed needed: 1.1138112013804011


## Step 10 — Adjust Persian audio duration

In this step, I used FFmpeg to speed up the generated Persian audio.

The `atempo` filter changes the audio tempo without changing the pitch too much. Since the generated audio was longer than the original video, I used the calculated speed value to reduce the audio duration.

The adjusted audio file is saved as:

`/content/persian_ai_voice_27sec.mp3`

This version is closer to the original video duration.

In [ ]:
audio_path = "/content/persian_ai_voice.mp3"
fixed_audio = "/content/persian_ai_voice_27sec.mp3"

speed = 30.984 / 27.818

!ffmpeg -y \
-i "$audio_path" \
-filter:a "atempo=$speed" \
"$fixed_audio"

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

## Step 11 — Verify the adjusted audio duration

After speeding up the Persian audio, I checked the duration again.

The original video duration is about 27.818 seconds.  
The adjusted Persian audio duration is about 27.864 seconds.

These two durations are very close, so the audio is now suitable for replacing the original English audio in the video.

In [ ]:
import subprocess

def get_duration(file):
    result = subprocess.run(
        [
            "ffprobe", "-v", "error",
            "-show_entries", "format=duration",
            "-of", "default=noprint_wrappers=1:nokey=1",
            file
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )
    return float(result.stdout.strip())

print("Video duration:", get_duration("/content/202606062328.mp4"))
print("Fixed audio duration:", get_duration("/content/persian_ai_voice_27sec.mp3"))

Video duration: 27.818
Fixed audio duration: 27.864


## Step 12 — Preview the adjusted Persian audio

In this step, I played the adjusted Persian audio inside Colab.

This is an important quality check before creating the final video. I checked that the voice still sounded natural after the tempo adjustment and that the audio duration was close to the original video duration.

In [ ]:
from IPython.display import Audio

Audio("/content/persian_ai_voice_27sec.mp3")

## Step 13 — Replace the original English audio with Persian AI voice

In this step, I created the final video.

I used the original video file and replaced its original English audio with the adjusted Persian AI voice. The video stream is copied from the original file, and the new Persian audio stream is added.

The `-map 0:v:0` command selects the video from the first input file.  
The `-map 1:a:0` command selects the audio from the second input file.  
The `-c:v copy` command keeps the original video quality without re-encoding.  
The `-c:a aac` command converts the audio to a format compatible with MP4.  
The `-shortest` option makes sure the final output stops when the shortest stream ends.

The final output is saved as:

`/content/final_persian_ai_video.mp4`

In [ ]:
video_path = "/content/202606062328.mp4"
fixed_audio = "/content/persian_ai_voice_27sec.mp3"
output_video = "/content/final_persian_ai_video.mp4"

!ffmpeg -y \
-i "$video_path" \
-i "$fixed_audio" \
-map 0:v:0 \
-map 1:a:0 \
-c:v copy \
-c:a aac \
-shortest \
"$output_video"

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

## Step 14 — Preview the final Persian video

In this step, I displayed the final video inside Colab.

This final preview allows me to check that the original English audio has been removed and replaced with the Persian AI-generated narration.

I also checked that the Persian voice is synchronized with the video and that the output plays correctly.

In [ ]:
from IPython.display import Video

Video("/content/final_persian_ai_video.mp4", embed=True)

## Step 15 — Download the final output video

After confirming that the final video works correctly, I downloaded it from Colab.

The downloaded file contains the original educational video with the English audio replaced by Persian AI-generated narration.

In [ ]:
from google.colab import files

files.download("/content/final_persian_ai_video.mp4")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Final conclusion

In this notebook, I tested different approaches for Persian narration and video audio replacement.

The Coqui XTTS voice cloning method was not suitable because Persian/Farsi was not properly supported, and using Arabic settings caused incorrect pronunciation.

The final successful method used `edge-tts`, which supports Persian/Iranian voices. I generated Persian narration, adjusted the audio duration to match the original video, and replaced the English audio using FFmpeg.

The final output is a Persian AI-narrated version of the original video.